In [1]:
import polars as pl
import numpy as np

from ohlc_dss_model.data import (
    load_parquet, remove_incomplete_days, write_parquet,
    intraday_session_tagging, session_tagging, 
    filter_valid_sessions
)

from ohlc_dss_model.features import(
    aggregate_sessions, yang_zhang,
    PRE_NY_SPEC, FULL_DAY_SPEC,
    calculate_excursion_bands, assign_direction,
    detect_pivots, pivot_extraction, build_pivot_features,
    build_event_table, encode_news_context, inspect_event_table
)

from ohlc_dss_model.utils import (
    convert_to_timezone
)

import os
from dotenv import load_dotenv

from ohlc_dss_model.config import config

from datetime import date

In [2]:
raw_data = load_parquet()
raw_data = convert_to_timezone(raw_data)
raw_data = session_tagging(raw_data)
raw_data = intraday_session_tagging(raw_data)
raw_data = remove_incomplete_days(raw_data)

aggregated_data = aggregate_sessions(raw_data)
aggregated_data = filter_valid_sessions(aggregated_data)

aggregated_data = aggregated_data.with_columns(
    pl.col("O_Asia").alias("O_Ref")
)

aggregated_data = yang_zhang(aggregated_data, FULL_DAY_SPEC, mode="historical")
aggregated_data = yang_zhang(aggregated_data, PRE_NY_SPEC, mode="today")

aggregated_data = assign_direction(aggregated_data)
aggregated_data = calculate_excursion_bands(aggregated_data)

pivots_data = detect_pivots(raw_data, n=1)
pivots_data = pivot_extraction(pivots_data)
pivots_data = build_pivot_features(pivots_data, aggregated_data)

In [3]:
print(pivots_data.columns)
pivots_data.filter(pl.col("Session") == date(2026, 2, 4)).select([
    "DateTime", "delta_Pi_k", "delta_b_k", "Speed_k", "Dir_k", "Turn_k", "P_k", "Pi_k"
]).head(8)

['DateTime', 'Intraday_Session', 'Session', 'P_k', 's_k', 'O_London', 'O_New York', 'O_Asia', 'H_London', 'H_New York', 'H_Asia', 'L_London', 'L_New York', 'L_Asia', 'C_London', 'C_New York', 'C_Asia', 'O_Ref', 'Sigma_Historical', 'Sigma_Today', 'Z_Body', 'Z_Sigma', 'Tau', 'Direction', '_delta_t', 'Band_AE_Pos_Center', 'Band_AE_Neg_Center', 'Band_FE_Pos_Center', 'Band_FE_Neg_Center', 'Band_AE_Neg_Upper', 'Band_AE_Neg_Lower', 'Band_AE_Pos_Upper', 'Band_AE_Pos_Lower', 'Band_FE_Neg_Upper', 'Band_FE_Neg_Lower', 'Band_FE_Pos_Upper', 'Band_FE_Pos_Lower', 'Pi_k', 'Sigma_Price', 'Delta_FE_Pos', 'Delta_AE_Pos', 'Delta_FE_Neg', 'Delta_AE_Neg', 'State_AE_Neg', 'State_AE_Pos', 'State_FE_Neg', 'State_FE_Pos', 'delta_Pi_k', 'delta_b_k', 'Speed_k', 'Dir_k', 'Turn_k']


DateTime,delta_Pi_k,delta_b_k,Speed_k,Dir_k,Turn_k,P_k,Pi_k
"datetime[ns, America/New_York]",f64,i16,f64,i32,i8,f64,f64
2026-02-03 19:00:00 EST,0.0,0,0.0,0,0,25435.75,0.160788
2026-02-03 19:30:00 EST,-0.199944,1,-0.199944,-1,0,25375.75,-0.039156
2026-02-03 21:00:00 EST,0.326576,3,0.108859,1,1,25473.75,0.28742
2026-02-03 21:30:00 EST,-0.247431,1,-0.247431,-1,1,25399.5,0.039989
2026-02-03 22:30:00 EST,0.174118,2,0.087059,1,1,25451.75,0.214107
2026-02-03 23:30:00 EST,-0.012497,2,-0.006248,-1,1,25448.0,0.20161
2026-02-04 00:30:00 EST,-0.130797,2,-0.065398,-1,0,25408.75,0.070814
2026-02-04 02:30:00 EST,0.351569,4,0.087892,1,1,25514.25,0.422382


***
### **Dataset Contract**
This section will be utilized the transform the already engineered feature datas into a dataset for transformer.

Before moving on we will be defining which features are to be included in the dataset this is done for us not to have any headache later

In [4]:
# Defining pivot column whitelists

# configurable in config.py
MAX_PIVOT = 27

PIVOT_NUMERIC_WHITELIST = [
    "Pi_k",
    'Delta_FE_Pos', 'Delta_AE_Pos', 'Delta_FE_Neg', 'Delta_AE_Neg',
    'State_AE_Neg', 'State_AE_Pos', 'State_FE_Neg', 'State_FE_Pos',
    "delta_Pi_k", "abs_delta_Pi_k", "delta_b_k", "Speed_k", "Dir_k", "Turn_k",
]

PIVOT_CATEGORICAL_WHITELIST = ["s_k", "Intraday_Session"]

In [5]:
_df = aggregated_data.with_columns(
    pl.col("Sigma_Historical").shift(1).alias("Sigma_Historical_Shifted")
)

#### **Normalizing Session OHLC Data**
Since we are feeding it into the context hence we will need to normalize it so it is consistent across all regime

$$\sigma_{Price} = \sigma_{YZ}(d-1) * O_Ref$$
$$\tilde{X}_{d}^{(s)} = \frac{X_{d}^{(s)} - O_{Ref,d}}{\sigma_{\text{Price}}}$$

where $X \in \{H, L, C, O\}$ and $s \in \{\text{Asia},\ \text{London}\}$.

In [6]:
sigma_price = pl.col("Sigma_Historical_Shifted") * pl.col("O_Ref")
_df = _df.with_columns([
    ((pl.col("H_Asia") - pl.col("O_Ref")) / sigma_price).alias("H_Asia_Normalized"),
    ((pl.col("L_Asia") - pl.col("O_Ref")) / sigma_price).alias("L_Asia_Normalized"),
    ((pl.col("C_Asia") - pl.col("O_Ref")) / sigma_price).alias("C_Asia_Normalized"),
    ((pl.col("O_London") - pl.col("O_Ref")) / sigma_price).alias("O_London_Normalized"),
    ((pl.col("H_London") - pl.col("O_Ref")) / sigma_price).alias("H_London_Normalized"),
    ((pl.col("L_London") - pl.col("O_Ref")) / sigma_price).alias("L_London_Normalized"),
    ((pl.col("C_London") - pl.col("O_Ref")) / sigma_price).alias("C_London_Normalized"),
])

***
#### **Economic Event Encoding**

For each trading day $d$, three event context features are constructed capturing the macro event weight on the preceding, current, and following calendar day:

$$\mathbf{n}_d = \left[\, e_{d-1},\ e_d,\ e_{d+1} \,\right] \in \{0, 1, 2, 3\}^3$$

where the daily event weight $e_t$ is defined as the maximum impact weight across all qualifying USD events on day $t$:

$$e_t = \max_{i \in \mathcal{E}_t} w_i, \qquad e_t = 0 \text{ if } \mathcal{E}_t = \emptyset$$

with impact weight tiers:

$$w_i = \begin{cases} 3 & \text{ultra-high: CPI, Core CPI, NFP, FOMC} \\ 2 & \text{high: Unemployment Claims, Average Hourly Earnings} \\ 1 & \text{medium: PPI, Core PPI, ADP, ISM Manufacturing,} \\ & \phantom{\text{medium: }} \text{ISM Services, JOLTs, Core Retail Sales} \\ 0 & \text{no qualifying event} \end{cases}$$

In [7]:
# load_dotenv()
# event_table = build_event_table(
#     aggregated_data.select(pl.col("Session").min()).item(),
#     aggregated_data.select(pl.col("Session").max()).item(),
#     os.getenv("ALFRED_API_KEY")
# )
# write_parquet(event_table, "event_table")

In [8]:
inspect = inspect_event_table(os.getenv("ALFRED_API_KEY"), date(2016, 1, 5), date(2016, 2, 10))
print(inspect.head(10))

  [ok] FOMC_SCRAPED         (US Federal Funds Rate): 0 meetings
shape: (10, 4)
┌────────────┬───────────────┬────────────────────────────────┬──────────┐
│ Session    ┆ series_id     ┆ name                           ┆ e_weight │
│ ---        ┆ ---           ┆ ---                            ┆ ---      │
│ date       ┆ str           ┆ str                            ┆ i64      │
╞════════════╪═══════════════╪════════════════════════════════╪══════════╡
│ 2016-01-06 ┆ ISM_SVC       ┆ US ISM Services PMI            ┆ 1        │
│ 2016-01-07 ┆ ICSA          ┆ US Unemployment Claims         ┆ 2        │
│ 2016-01-08 ┆ CES0500000003 ┆ US Average Hourly Earnings m/m ┆ 2        │
│ 2016-01-08 ┆ MANEMP        ┆ US ISM Manufacturing PMI       ┆ 1        │
│ 2016-01-08 ┆ PAYEMS        ┆ US Non-Farm Employment Change  ┆ 3        │
│ 2016-01-12 ┆ JTSJOL        ┆ US JOLTS Job Openings          ┆ 1        │
│ 2016-01-14 ┆ ICSA          ┆ US Unemployment Claims         ┆ 2        │
│ 2016-01-15 ┆ WPSFD4

In [9]:
event_table = load_parquet(config.data.event_path)
aggregated_data = encode_news_context(aggregated_data, event_table)

In [10]:
event_table.head(5)

Session,e_weight
date,i8
2016-01-06,1
2016-01-07,2
2016-01-08,3
2016-01-12,1
2016-01-14,2


In [11]:
aggregated_data.filter(pl.col("Session") == date(2026, 2, 12)).select(["Session", "e_yesterday", "e_today", "e_tomorrow"]).head(10)

Session,e_yesterday,e_today,e_tomorrow
date,i8,i8,i8
2026-02-12,3,2,3


***

The context table $\mathcal{C}$ is indexed by trading day $d \in \mathcal{D}$:

$$\mathcal{C}_d = \left[\,\tilde{H}_d^{A},\ \tilde{L}_d^{A},\ \tilde{C}_d^{A},\ \tilde{O}_d^{L},\ \tilde{H}_d^{L},\ \tilde{L}_d^{L},\ \tilde{C}_d^{L},\ \sigma_d,\ \sigma_{d-1}\,\right] \in \mathbb{R}^{9}$$

where $d$ corresponds to the `Session` date key used for alignment with pivot sequences and targets.

In [ ]:
# Columns that will go in context table

CONTEXT_WHITELIST = [
    "Sigma_Today", "Sigma_Historical_Shifted",
    "e_yesterday", "e_today", "e_tomorrow",
    "H_Asia_Normalized", "L_Asia_Normalized", "C_Asia_Normalized",
    "O_London_Normalized", "H_London_Normalized",
    "L_London_Normalized", "C_London_Normalized",
]

In [13]:
ctx_df = (
    _df.select(["Session", *CONTEXT_WHITELIST])
    .drop_nulls(["Session", *CONTEXT_WHITELIST])
    .sort("Session")
)

In [14]:
print(ctx_df.columns)

['Session', 'Sigma_Today', 'Sigma_Historical_Shifted', 'H_Asia_Normalized', 'L_Asia_Normalized', 'C_Asia_Normalized', 'O_London_Normalized', 'H_London_Normalized', 'L_London_Normalized', 'C_London_Normalized']


In [15]:
ctx_df.filter(pl.col("Session") == date(2026, 2, 4))

Session,Sigma_Today,Sigma_Historical_Shifted,H_Asia_Normalized,L_Asia_Normalized,C_Asia_Normalized,O_London_Normalized,H_London_Normalized,L_London_Normalized,C_London_Normalized
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-02-04,0.00523,0.01182,0.422382,-0.084976,0.384893,0.382393,0.406553,-0.109969,0.022494
